
### Prerequisites

1.  **A running Google Cloud Composer Environment:** You must have already created a Composer environment in your Google Cloud Project (check `03_environment_setup_cli.ipynb`).
2.  **Your DAG file(s):** These are Python `.py` files containing your Airflow DAG definition (examples are provided in `code_examples`).
3.  **Permissions:** Ensure your Google Cloud account has the necessary IAM permissions (e.g., `Composer Environment and Storage Object Administrator` or at least permissions to upload objects to the Composer bucket, check also check `03_environment_setup_cli.ipynb` for more).
4.  **Google Cloud SDK (optional but recommended):** For command-line operations (`gcloud` and `gsutil`).

### Step 1: Locate Your Composer Environment's DAGs Bucket

Every Cloud Composer environment has a dedicated Cloud Storage bucket where your DAG files are stored. Airflow in Composer monitors this bucket's `/dags` folder for new or updated DAGs.

**Via Google Cloud Console:**

1.  Go to the Google Cloud Console: [https://console.cloud.google.com/](https://console.cloud.google.com/)
2.  Navigate to **Composer** (or search for "Composer" in the search bar).
3.  On the "Environments" page, you'll see a list of your Composer environments.
4.  Find the row corresponding to your environment. In the "DAGs folder" column, click the **"DAGs" link**. This will open the Cloud Storage bucket directly to the `/dags` folder.
![image.png](../artifacts/04_composer_dags_bucket_link.png)



**Via `gcloud` CLI:**

1.  Open your terminal or Cloud Shell.

2.  Run the following command to get the DAGs bucket prefix:


In [ ]:
# Retrieve the Cloud Storage /dags bucket URI for your Composer environment
    gcloud composer environments describe YOUR_ENVIRONMENT_NAME \
        --location YOUR_LOCATION \
        --format="get(config.dagGcsPrefix)"


The output will be something like `gs://your-bucket-name/dags`. This is the path you'll upload your DAGs to.


### Step 2: Upload Your DAG Code to the DAGs Bucket

This is how Airflow discovers and loads your DAGs.

**Method A: Using Google Cloud Console (Recommended for manual uploads)**

1.  Follow Step 1 to navigate to your Composer environment's `/dags` folder in Cloud Storage.
2.  Click the **"Upload files"** button.
3.  Select your DAG Python file (e.g., `bq_etl_dag.py`) from your local machine using the file browser dialog.
4.  Confirm the upload.

**Method B: Using `gsutil` CLI (Recommended for scripting/automation)**

1.  Open your terminal or Cloud Shell.

2.  Use the `gsutil cp` command to copy your DAG file:


In [ ]:
# Copy a single DAG file from your local machine to the Composer /dags bucket
gsutil cp 04_dataproc_serverless_etl/dags/bq_etl_dag.py gs://YOUR_COMPOSER_DAGS_BUCKET/dags/


**For multiple DAGs or a DAGs directory:**


In [ ]:
# Synchronize an entire local DAGs directory with the Composer /dags bucket
gsutil rsync -d -r 04_dataproc_serverless_etl/dags/ gs://YOUR_COMPOSER_DAGS_BUCKET/dags/
#  *`rsync` is useful for syncing entire folders. 
# `-d` deletes extra files in the destination, and `-r` is for recursive.*


**What happens next:**

Once the DAG file is in the `/dags` folder of your Composer bucket:

  * The Composer environment's Airflow components (scheduler, workers) automatically sync this bucket.
  * The Airflow scheduler will detect the new DAG file, parse it, and add it to the Airflow metadata database.
  * This process typically takes 3-5 minutes for the DAG to appear in the Airflow UI. For updates, it's recommended to pause the DAG in the UI before uploading, then un-pause after verification.

### Step 3: Verify DAG Appearance in Airflow UI

1.  In the Google Cloud Console, go to your **Composer environment's page**.
2.  Click on the **"Airflow web interface"** link. This will open the Airflow UI in a new browser tab.
3.  In the Airflow UI, navigate to the **"DAGs"** tab (usually the default view).
4.  You should see your `bq_etl_dag.py` (or whatever you named your DAG's `dag_id`) listed.
5.  By default, newly uploaded DAGs might be in a "Paused" state. Toggle the switch next to your DAG's name to **"Unpause"** it.

![image.png](../artifacts/04_airflow_ui_unpause_dag.png)

### Step 4: Launch (Trigger) Your DAG

Once your DAG is unpaused, Airflow will automatically schedule it based on its `schedule_interval` (if defined in your DAG code). However, you often want to trigger it manually for testing or immediate execution.

**Method A: Manually Trigger from Airflow UI (Most Common)**

1.  In the Airflow UI, on the "DAGs" tab, find your DAG.
2.  Click the **"Play" icon** (Trigger DAG) located to the right of your DAG's name.
3.  A "Trigger DAG" dialog will appear. You can optionally provide a "conf" JSON object if your DAG expects configuration parameters (`dag_run.conf`).
4.  Click **"Trigger"**.

**Method B: Trigger from `gcloud` CLI (Useful for automation/scripting)**

1.  Open your terminal or Cloud Shell.

2.  Use the `gcloud composer environments run` command with the `dags trigger` subcommand:


In [ ]:
# Trigger a DAG run manually via the gcloud Composer CLI
gcloud composer environments run YOUR_ENVIRONMENT_NAME \
    --location YOUR_LOCATION \
    dags trigger YOUR_DAG_ID


**To pass configuration:**


In [ ]:
# Trigger a DAG run via the gcloud CLI and pass runtime JSON configuration
    gcloud composer environments run YOUR_ENVIRONMENT_NAME \
        --location YOUR_LOCATION \
        dags trigger YOUR_DAG_ID --conf '{"key": "value", "another_key": "another_value"}'


**What happens after triggering:**

  * A "DAG Run" instance will be created for your DAG.
  * You can monitor the progress of your DAG run in the Airflow UI under the "Graph View," "Gantt Chart," or "Task Instances" tabs for your specific DAG. You'll see tasks transition from "scheduled" to "running" to "success" (or "failed") (check `01_airflow_fundamentals.ipynb` for more info)
  * You can view task logs directly from the Airflow UI to debug any issues.

This comprehensive guide covers the typical workflow for deploying and running your Apache Airflow DAGs on Google Cloud Composer.


### Example `bq_etl_dag.py` and env variable

To explain better the env vars, described in `01_airflow_fundamentals.ipynb`, Airflow Variables (`var.value.YOUR_VARIABLE_NAME`) are key-value pairs stored in Airflow's metadata database that you can manage through the Airflow UI or CLI. In the dag proposed for this training, we have set an env vars without defining it, therefore,  if you trigger the dag and check the logs you should have this error: 
```
KeyError: 'Variable airflow_worker_gcp_service_account does not exist'
```

This error occurs because your `DataprocCreateBatchOperator` is trying to retrieve the service account email for your Spark job from an **Airflow Variable** named `airflow_worker_gcp_service_account`, but this Variable is not found in your Airflow environment.

The line causing this is:

```python
"service_account": "{{ var.value.airflow_worker_gcp_service_account }}"
```


#### How to Fix the Missing Airflow Variable

You can define `airflow_worker_gcp_service_account` (and `SUBNET_NAME`) either via the **Airflow UI** or the **`gcloud` CLI**:

1. **Via Airflow Web UI:**
   * Navigate to **Admin > Variables** in the top menu bar.
   * Click the **+** (Add a new record) button.
   * Set **Key** to `airflow_worker_gcp_service_account` and **Val** to `example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com`.
   * Click **Save**.

2. **Via `gcloud` CLI:**
```bash
gcloud composer environments run YOUR_ENVIRONMENT_NAME \
    --location YOUR_LOCATION \
    variables set -- airflow_worker_gcp_service_account "example-workshop-composer-sa@poc-example-ds.iam.gserviceaccount.com"
```


### How to check the logs and identify an error

### How to Check Logs and Identify an Error in Airflow (Google Cloud Composer)

When a task in your Airflow DAG fails, the first place to look for clues is its specific log file. Airflow's web UI provides direct access to these logs.

**Step 1: Access the Airflow Web Interface**

1.  Go to the Google Cloud Console: [https://console.cloud.google.com/](https://console.cloud.google.com/)
2.  Navigate to **Composer** (or search for "Composer" in the search bar).
3.  On the "Environments" page, find your Composer environment.
4.  Click on the **"Airflow web interface"** link. This will open the Airflow UI in a new browser tab.

**Step 2: Navigate to the DAGs View**

1.  Once in the Airflow UI, you'll usually land on the **"DAGs"** list view.
2.  Locate the DAG that contains the failing task. You can use the search bar or filters if you have many DAGs.

**Step 3: Open the DAG's Graphical or Grid View**

There are two primary views to see the status of your DAG runs and tasks:

  * **Grid View (Recommended for overall DAG run status):**

      * Click on the **name of your DAG**.
      * This view shows a grid where each row is a task and each column is a DAG run. Failed tasks are typically highlighted in **red**.
      * Find the specific DAG run (column) and the specific task (row) that failed.

  * **Graph View (Recommended for understanding dependencies and failure paths):**

      * Click on the **name of your DAG**.
      * Then, click on the **"Graph" tab** (or "Graph View" in older versions).
      * This view visually represents your DAG's tasks and their dependencies. Failed tasks will be colored **red**. This is very helpful to see which upstream task failed and potentially caused downstream failures.

**Step 4: Click on the Specific Failing Task Instance**

Whether you're in Grid View or Graph View:

1.  **Click on the red-colored box/node** that represents the failed task instance.
2.  A details panel will open on the right side of the screen.
3.  Select the **Logs** tab to inspect the task execution output, stack traces, and link directly to the **Dataproc Serverless Batch logs** in Cloud Logging.idebar will appear, showing details about that task instance.

**Step 5: Access the "Log" Tab**

1.  In the task instance details pop-up/sidebar, you will see several tabs (e.g., "Details", "Rendered", "XComs", "Log").
2.  Click on the **"Log" tab**.

**Step 6: Read and Interpret the Logs**

The "Log" tab will display the standard output (stdout) and standard error (stderr) generated by your task's execution.

**How to identify an error:**

  * **Look for `ERROR` or `CRITICAL` keywords:** These are explicit error messages.
  * **Look for `Traceback`:** This is the most common indicator of a Python code error. A traceback will show you the exact line number and file where the error occurred, along with the call stack that led to it.
      * Example: `Traceback (most recent call last): File "<string>", line 1, in <module> ... ZeroDivisionError: division by zero`
  * **Identify the final exit code:** For `BashOperator` tasks, a non-zero exit code (e.g., `exit code 1`) indicates a failure. The log messages leading up to this will usually explain why.
  * **Search for keywords:** If you know what your task was supposed to do, search for related keywords in the logs to narrow down the problem.
  * **Check timestamps:** Pay attention to the timestamps in the logs to understand the sequence of events leading to the failure.
  * **Resource Exhaustion:** Sometimes, tasks fail without a clear code error in the log, but rather due to out-of-memory (OOM) errors or other resource constraints. In such cases, you might see messages related to "Killed" or "Terminated" by the operating system, or the log might simply cut off abruptly. For such issues, you might need to check Cloud Logging for the underlying worker logs (see "Alternative: Cloud Logging" below).

![image.png](../artifacts/04_airflow_ui_task_logs.png)

### Alternative: Cloud Logging (Stackdriver Logs)

While the Airflow UI is excellent for task-specific logs, Cloud Composer also integrates deeply with Google Cloud's **Cloud Logging (formerly Stackdriver Logging)**. This is crucial for:

  * **System-level errors:** Issues with Airflow components themselves (scheduler, webserver, workers), rather than just your DAG task code.
  * **DAG parsing errors:** If your DAG doesn't even appear in the Airflow UI or shows as "Broken DAG", the error is likely during parsing, and you'll find it in Cloud Logging.
  * **Resource issues:** As mentioned, OOM errors or worker issues that terminate tasks without a clean Python traceback.
  * **Historical logs:** Cloud Logging retains logs for longer periods (default 30 days, configurable) than the Airflow UI might display.

**How to check Cloud Logging:**

1.  Go to the Google Cloud Console: [https://console.cloud.google.com/](https://www.google.com/search?q=https://console.cloud.com/)
2.  Navigate to **Logging \> Logs Explorer** (or search for "Logs Explorer").
3.  In the "Resource" dropdown, select `Kubernetes Container`.
4.  Then, in the "Container Name" or "Pod Name" dropdown, select relevant Airflow components:
      * `airflow-worker`: For task execution logs (often duplicates of what's in the UI, but can have more context).
      * `airflow-scheduler`: For DAG parsing errors, scheduling issues, and general Airflow operations.
      * `airflow-webserver`: For UI-related issues or errors when interacting with the web interface.
      * `airflow-triggerer` (if enabled): For logs related to deferrable operators.
5.  You can also filter by `logName` (e.g., `projects/YOUR_PROJECT_ID/logs/airflow-scheduler`) or by searching for your DAG ID, task ID, or specific error messages.

### Solution: You need to create the missing Airflow Variable.

**Option 1 (Recommended): Create the Airflow Variable in the UI**

This is the standard and most flexible way to manage such configurations.

1.  **Go to your Airflow UI.** (Access your Composer environment's Airflow UI URL).
2.  Navigate to **Admin** -\> **Variables**.
3.  Click the **+** (plus) button or **"Create"** to add a new Variable.
4.  For **Key**, enter: `airflow_worker_gcp_service_account`
5.  For **Value**, enter the full email address of the Google Cloud Service Account that your Dataproc Serverless jobs should run as. This is typically the default Compute Engine service account or a dedicated service account you created for Dataproc jobs.
      * **Example Value:** `your-project-id@appspot.gserviceaccount.com`
      * **Example Value (Custom SA):** `my-dataproc-sa@your-project-id.iam.gserviceaccount.com`
6.  Click **"Save"**.

Once this variable is created, Airflow will be able to retrieve its value during DAG parsing and execution, resolving the `KeyError`.

![image-2.png](../artifacts/04_airflow_ui_trigger_dag.png)



 


### Retrigger manually a DAG
Click the "Trigger DAG" (Play) Button
- On the right-hand side of the row for your target DAG, you will see a series of icons.
- Look for the "Play" button icon (it often looks like a triangular play symbol, similar to what you'd find on a video player).
- Click this "Play" button. 
![image.png](../artifacts/04_airflow_ui_retrigger_dag.png)
